In [1]:
!python -m pip install --upgrade pip --default-timeout=300

In [2]:
!python -m pip install mlflow --timeout 1200 --retries 10 --no-cache-dir

In [3]:
!pip install awscli

In [4]:
!pip install boto3

In [5]:
!pip install python-dotenv

In [32]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [33]:
!aws configure set aws_access_key_id os.getenv("aws_access_key_id")
!aws configure set aws_secret_access_key os.getenv("aws_secret_access_key")
!aws configure set region "ap-south-1"

In [34]:
import mlflow

mlflow.set_tracking_uri("http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/")

In [35]:
mlflow.set_experiment("Exp 2 - BOW vs TFIDF")

<Experiment: artifact_location='s3://mlflow-bucket-2181/2', creation_time=1789716511753, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1789716511753, lifecycle_stage='active', name='Exp 2 - BOW vs TFIDF', tags={}, trace_location=None, workspace='default'>

In [36]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [37]:
df = pd.read_csv("cleaned_df.csv")
df.shape

(36793, 3)

In [38]:
df.columns

Index(['Unnamed: 0', 'clean_comment', 'category'], dtype='object')

In [39]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [40]:
df

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1
...,...,...
36788,jesus,0
36789,kya bhai pure saal chutiya banaya modi aur jab...,1
36790,downvote karna tha par upvote hogaya,0
36791,haha nice,1


In [41]:
df.isna().sum()

clean_comment    131
category           0
dtype: int64

In [42]:
df.dropna(inplace=True)

In [43]:
df.isna().sum()

clean_comment    0
category         0
dtype: int64

In [44]:
df['category'].value_counts()

category
 1    15770
 0    12644
-1     8248
Name: count, dtype: int64

In [45]:
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X = df['clean_comment']
    X = vectorizer.fit_transform(X)
    Y = df['category']

    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=.2, random_state=42, stratify=Y)


    with mlflow.start_run() as run:
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(x_train, y_train)

        y_pred = model.predict(x_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion matrix: {vectorizer_name}, {ngram_range}")

        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        mlflow.sklearn.log_model(
        model,
        name=f"RandomForest_model_{vectorizer_name}_{ngram_range}",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

In [47]:
ngram_ranges = [(1,1), (1,2), (1,3)]
max_features = 5000

for ngram_range in ngram_ranges:
    # BoW experiment
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    # TF-IDF experiment
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")

🏃 View run languid-jay-549 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/2/runs/fbd2206bacdf45768b048ac044f74921
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/2
🏃 View run powerful-grouse-472 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/2/runs/d9e5ef3e3f1746169b00f44a4dde3b34
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/2
🏃 View run languid-mare-960 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/2/runs/d2a50787902941e89ab52cbc429f571e
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/2
🏃 View run enchanting-ox-659 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/2/runs/bf015d42543e46e6810f2d4d30ad0ae1
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments